# 🏎️ F1 BI — Dashboard Streamlit
## Proyecto Final · Módulo 4: Inteligencia de Negocios y SQL Avanzado

Este notebook clona el repositorio, configura el entorno y lanza el dashboard interactivo usando **Streamlit** expuesto con un túnel **ngrok**.

### Visualizaciones incluidas
| # | Visualización | Técnica SQL |
|:---:|---|---|
| 1 | Campeonato — evolución de puntos por ronda | Window Functions |
| 2 | Estrategia — pit stops y posición de salida | Agregaciones |
| 3 | Dominancia — eficiencia de constructores por era | CTEs anidados |
| 4 | Circuitos — posición mediana de ganadores | PERCENTILE_CONT |
| 5 | Edad — evolución histórica de ganadores | DATE_PART + LAG() |

### Repositorio
`https://github.com/PercastreMarco/Analisis-Formula-1-World-Championship-1950---2024-`

## 1. Instalación de dependencias

In [81]:
!pip install streamlit plotly sqlalchemy psycopg2-binary pyngrok -q
print('✅ Dependencias instaladas')

✅ Dependencias instaladas


## 2. Variables de entorno

Las credenciales de Aurora se leen desde los **Secrets de Colab** (ícono de llave 🔑 en el panel izquierdo) y se pasan como variables de entorno para que `dashboard.py` las lea con `os.getenv()`.

In [104]:
import os
from google.colab import userdata

os.environ['F1_HOST'] = userdata.get('F1_HOST')
os.environ['F1_DATA'] = userdata.get('F1_data')
os.environ['F1_USER'] = userdata.get('AURORA_USER')

print('✅ Variables de entorno configuradas')
print(f'   Host: {os.environ["F1_HOST"][:25]}...')

✅ Variables de entorno configuradas
   Host: f1data.cluster-cuzcmcs3sw...


## 3. Clonar repositorio y posicionarse en `dashboard/`

Clona el repositorio completo desde GitHub y cambia el directorio de trabajo a `dashboard/` donde vive `Dashboard.py`. Así Streamlit lo encuentra directamente.

In [102]:
import os

REPO_URL  = 'https://github.com/PercastreMarco/Analisis-Formula-1-World-Championship-1950---2024-.git'
REPO_DIR  = '/content/f1_repo'
DASH_DIR  = f'{REPO_DIR}/dashboard'

# Clonar solo si no existe ya
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repositorio ya clonado — actualizando...')
    !git -C {REPO_DIR} pull

# Cambiar al directorio dashboard/
os.chdir(DASH_DIR)

# Verificar
print(f'\n📂 Directorio actual : {os.getcwd()}')
print(f'📄 Archivos en dashboard/ : {os.listdir(".")}')
print(f'\n✅ Dashboard.py encontrado: {os.path.exists("Dashboard.py")}')

Repositorio ya clonado — actualizando...
Already up to date.

📂 Directorio actual : /content/f1_repo/dashboard
📄 Archivos en dashboard/ : ['03_dashboard.ipynb', 'Dashboard.py']

✅ Dashboard.py encontrado: True


## 4. Configurar ngrok

**ngrok** crea un túnel HTTPS público que expone el servidor Streamlit (que corre en `localhost:8501` dentro de Colab) a tu navegador.

**Obtener token gratuito:** https://dashboard.ngrok.com/get-started/your-authtoken  
Guárdalo en los Secrets de Colab con el nombre `NGROK_TOKEN`.

In [96]:
from pyngrok import ngrok

# Pegar el token directamente (solo para desarrollo local)
ngrok.set_auth_token("3F0xdcoG4DETC60PQP5dOsig1dq_4E6bghduM7Y6pnZ6Qpa3p")
print('✅ ngrok configurado correctamente')

✅ ngrok configurado correctamente


## 5. Lanzar el dashboard

Esta celda:
1. Mata procesos previos de Streamlit y ngrok
2. Lanza `Dashboard.py` en el puerto 8501
3. Espera con `socket` hasta que el puerto esté realmente abierto
4. Abre el túnel ngrok **solo cuando Streamlit está listo**
5. Imprime la URL pública

> El proceso tarda ~15 segundos. Si la URL da error al principio, espera y recarga.

In [103]:
import subprocess, time, socket, os
from pyngrok import ngrok

# 1. Matar procesos previos
!pkill -f streamlit 2>/dev/null || True
ngrok.kill()
time.sleep(2)

# 2. Confirmar que estamos en el directorio correcto
print(f'📂 Directorio: {os.getcwd()}')
print(f'📄 Dashboard.py existe: {os.path.exists("Dashboard.py")}')

# 3. Lanzar Streamlit en background
proc = subprocess.Popen(
    ['streamlit', 'run', 'Dashboard.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# 4. Esperar hasta que el puerto 8501 responda (máx 90 segundos)
def puerto_abierto(host='localhost', port=8501, timeout=90):
    inicio = time.time()
    while time.time() - inicio < timeout:
        try:
            with socket.create_connection((host, port), timeout=2):
                return True
        except (ConnectionRefusedError, OSError):
            print('.', end='', flush=True)
            time.sleep(1)
    return False

print('\n⏳ Esperando que Streamlit inicie', end='')
if puerto_abierto():
    print(' ✅ Streamlit listo')
else:
    print(' ❌ Timeout — revisa el error abajo:')
    out, err = proc.communicate(timeout=5)
    print('STDOUT:', out.decode()[:500])
    print('STDERR:', err.decode()[:500])
    raise RuntimeError('Streamlit no pudo iniciar')

# 5. Abrir túnel ngrok SOLO cuando Streamlit está listo
tunnel = ngrok.connect(8501, proto='http')
url    = tunnel.public_url

print('\n' + '=' * 55)
print('  🏎️  F1 Analytics Dashboard')
print('=' * 55)
print(f'  URL pública : {url}')
print('  Abre esa URL en tu navegador')
print('  Para cerrar : ejecuta la celda de abajo')
print('=' * 55)

^C
📂 Directorio: /content/f1_repo/dashboard
📄 Dashboard.py existe: True

⏳ Esperando que Streamlit inicie.. ✅ Streamlit listo

  🏎️  F1 Analytics Dashboard
  URL pública : https://sanding-enjoyment-lapped.ngrok-free.dev
  Abre esa URL en tu navegador
  Para cerrar : ejecuta la celda de abajo


## 6. Detener el dashboard

Ejecuta esta celda cuando termines para liberar el túnel ngrok y el proceso Streamlit.

In [90]:
from pyngrok import ngrok

ngrok.kill()
!pkill -f streamlit 2>/dev/null || True
print('✅ Dashboard detenido correctamente')

^C
✅ Dashboard detenido correctamente


In [91]:
import os, time

# 1. Matar TODO lo relacionado a Streamlit
!pkill -9 -f streamlit 2>/dev/null || True
!fuser -k 8501/tcp 2>/dev/null || True
!fuser -k 8502/tcp 2>/dev/null || True
time.sleep(3)

# 2. Verificar que el puerto quedó libre
import socket
def puerto_libre(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) != 0

print(f"Puerto 8501 libre: {puerto_libre(8501)}")
print(f"Puerto 8502 libre: {puerto_libre(8502)}")

^C
/bin/bash: line 1: True: command not found
/bin/bash: line 1: True: command not found
Puerto 8501 libre: True
Puerto 8502 libre: True


## 7. Solución de problemas comunes

| Problema | Causa | Solución |
|---|---|---|
| `File does not exist: Dashboard.py` | Directorio incorrecto | Re-ejecuta la Celda 3 |
| `502 Bad Gateway` en la URL | Streamlit aún iniciando | Espera 10s y recarga |
| `ERR_NGROK_8012` | Puerto 8501 no responde | Re-ejecuta la Celda 5 |
| `AuthError` en ngrok | Token incorrecto o expirado | Verifica el Secret `NGROK_TOKEN` |
| Gráficas vacías | Credenciales Aurora incorrectas | Verifica los Secrets F1_HOST, F1_data, AURORA_USER |
| Colab se desconecta | Timeout de inactividad | Re-ejecuta desde Celda 2 |

In [87]:
import time

# Esperar unos segundos para que Streamlit procese
time.sleep(5)

# Leer el output del proceso
try:
    out, err = proc.communicate(timeout=3)
    print("STDOUT:", out.decode()[:2000])
    print("STDERR:", err.decode()[:2000])
except:
    # Si el proceso sigue vivo, leer sin bloquearlo
    import os
    out = proc.stdout.read1(2000).decode() if proc.stdout else ""
    err = proc.stderr.read1(2000).decode() if proc.stderr else ""
    print("STDOUT:", out)
    print("STDERR:", err)

STDOUT: 


STDERR: 2026-06-12 01:35:14.226 Port 8501 is not available

